# 08 — Topic Model
### Brand Sentiment Monitor

Trains a **BERTopic** model on all training texts to discover topics customers
discuss (sole durability, sizing, pricing, etc.).

No supervised labels needed. BERTopic uses sentence embeddings + HDBSCAN.

**Outputs:**
```
models/topic/                    saved BERTopic model
outputs/reports/topic_eval.json  topic inventory + coherence
outputs/visualizations/08_*.png  topic distribution charts
```

---
Run notebooks 00 → 07 first.

## 0. Setup

In [7]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import os, sys, json, ast, warnings, random
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

DRIVE_ROOT  = "/content/drive/MyDrive/brand-sentiment-monitor"
KAGGLE_PROC = os.path.join(DRIVE_ROOT, "data/kaggle/processed")
MODEL_OUT   = os.path.join(DRIVE_ROOT, "models/topic")
OUTPUTS_VIZ = os.path.join(DRIVE_ROOT, "outputs/visualizations")
OUTPUTS_RPT = os.path.join(DRIVE_ROOT, "outputs/reports")

for d in [MODEL_OUT, OUTPUTS_VIZ, OUTPUTS_RPT]:
    os.makedirs(d, exist_ok=True)

sys.path.insert(0, DRIVE_ROOT)
sys.path.insert(0, os.path.join(DRIVE_ROOT, "src"))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Paths set ✅")


Mounted at /content/drive
Paths set ✅


In [8]:
!pip install bertopic --quiet
!pip install sentence-transformers --quiet
!pip install umap-learn --quiet
!pip install hdbscan --quiet

In [9]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
print("BERTopic imported ✅")


BERTopic imported ✅


## 1. Load and Combine Training Texts

In [10]:
# Combine all three cleaned datasets for topic discovery
s140 = pd.read_csv(os.path.join(KAGGLE_PROC, "s140_clean.csv"))
ge   = pd.read_csv(os.path.join(KAGGLE_PROC, "goemotions_clean.csv"))
sem  = pd.read_csv(os.path.join(KAGGLE_PROC, "semeval_clean.csv"))

# Sample — BERTopic is memory-intensive; 100k texts is sufficient for good topics
s140_sample = s140["text"].dropna().astype(str).sample(50_000, random_state=SEED)
ge_sample   = ge["text"].dropna().astype(str).sample(30_000, random_state=SEED)
sem_texts   = sem["text"].dropna().astype(str)

all_texts = pd.concat([s140_sample, ge_sample, sem_texts], ignore_index=True)
all_texts = all_texts[all_texts.str.len() >= 15].reset_index(drop=True)

print(f"Combined texts: {len(all_texts):,}")
print(f"  from S140      : {len(s140_sample):,}")
print(f"  from GoEmotions: {len(ge_sample):,}")
print(f"  from SemEval   : {len(sem_texts):,}")


Combined texts: 84,548
  from S140      : 50,000
  from GoEmotions: 30,000
  from SemEval   : 4,582


## 2. Configure and Fit BERTopic

In [11]:
# Configure components
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")  # fast, good quality

umap_model  = UMAP(
    n_components=5, n_neighbors=15,
    min_dist=0.0, metric="cosine",
    random_state=SEED, low_memory=True,
)
hdbscan_model = HDBSCAN(
    min_cluster_size=50, min_samples=10,
    metric="euclidean", cluster_selection_method="eom",
    prediction_data=True,
)
vectorizer = CountVectorizer(
    stop_words="english",
    min_df=10, max_df=0.95,
    ngram_range=(1, 2),
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer,
    nr_topics="auto",
    calculate_probabilities=True,
    verbose=True,
)

print("BERTopic configured ✅")
print("Fitting on combined corpus (~10-20 min on CPU, ~3-5 min with GPU)...")
topics, probs = topic_model.fit_transform(all_texts.tolist())
print(f"\nFitting complete ✅")
print(f"  Topics discovered: {len(set(topics)) - (1 if -1 in topics else 0)}")
print(f"  Outlier posts (-1): {topics.count(-1):,} ({topics.count(-1)/len(topics)*100:.1f}%)")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-03-04 06:44:22,269 - BERTopic - Embedding - Transforming documents to embeddings.


BERTopic configured ✅
Fitting on combined corpus (~10-20 min on CPU, ~3-5 min with GPU)...


Batches:   0%|          | 0/2643 [00:00<?, ?it/s]

2026-03-04 06:45:02,349 - BERTopic - Embedding - Completed ✓
2026-03-04 06:45:02,349 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-04 06:47:12,231 - BERTopic - Dimensionality - Completed ✓
2026-03-04 06:47:12,235 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-04 06:51:10,237 - BERTopic - Cluster - Completed ✓
2026-03-04 06:51:10,238 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-03-04 06:51:11,873 - BERTopic - Representation - Completed ✓
2026-03-04 06:51:11,874 - BERTopic - Topic reduction - Reducing number of topics
2026-03-04 06:51:11,979 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-04 06:51:13,458 - BERTopic - Representation - Completed ✓
2026-03-04 06:51:13,467 - BERTopic - Topic reduction - Reduced number of topics from 216 to 161



Fitting complete ✅
  Topics discovered: 160
  Outlier posts (-1): 38,244 (45.2%)


## 3. Explore Topics

In [12]:
topic_info = topic_model.get_topic_info()
print(f"Topic inventory ({len(topic_info)} topics including outlier):")
print()
for _, row in topic_info.head(20).iterrows():
    if row["Topic"] == -1:
        print(f"  Topic -1  (outlier)  {row['Count']:>6,} posts")
        continue
    keywords = [w for w, _ in topic_model.get_topic(row["Topic"])][:6]
    print(f"  Topic {row['Topic']:>3}  {row['Count']:>6,} posts  {', '.join(keywords)}")


Topic inventory (161 topics including outlier):

  Topic -1  (outlier)  38,244 posts
  Topic   0   5,675 posts  team, game, season, win, players, year
  Topic   1   3,661 posts  sleep, bed, morning, night, work, good morning
  Topic   2   2,441 posts  food, eat, eating, hungry, lunch, chocolate
  Topic   3   1,644 posts  movie, book, episode, watch, watching, season
  Topic   4   1,548 posts  twitter, tweet, tweets, tweeting, goodnight, welcome
  Topic   5   1,476 posts  sick, flu, hurts, sore, feeling, feel
  Topic   6   1,272 posts  song, music, album, listening, songs, listen
  Topic   7   1,224 posts  does, girl, did, talk, looks, ask
  Topic   8   1,157 posts  school, exam, homework, graduation, exams, studying
  Topic   9   1,039 posts  concert, london, tour, la, chicago, england
  Topic  10     849 posts  money, bank, pay, price, sell, rich
  Topic  11     834 posts  iphone, phone, update, app, internet, mobile
  Topic  12     833 posts  hair, dress, shirt, wear, makeup, hat
  T

In [13]:
# Assign human-readable labels to top topics
# Inspect keywords and write short labels
topic_labels = {}
for topic_id in topic_info["Topic"].tolist():
    if topic_id == -1:
        continue
    keywords = [w for w, _ in topic_model.get_topic(topic_id)][:3]
    label    = "_".join(keywords)[:40] if keywords else f"topic_{topic_id}"
    topic_labels[topic_id] = label

print(f"Generated labels for {len(topic_labels)} topics:")
for tid, label in list(topic_labels.items())[:15]:
    print(f"  Topic {tid:>3}: {label}")


Generated labels for 160 topics:
  Topic   0: team_game_season
  Topic   1: sleep_bed_morning
  Topic   2: food_eat_eating
  Topic   3: movie_book_episode
  Topic   4: twitter_tweet_tweets
  Topic   5: sick_flu_hurts
  Topic   6: song_music_album
  Topic   7: does_girl_did
  Topic   8: school_exam_homework
  Topic   9: concert_london_tour
  Topic  10: money_bank_pay
  Topic  11: iphone_phone_update
  Topic  12: hair_dress_shirt
  Topic  13: rain_raining_sun
  Topic  14: cat_dog_dogs


In [15]:
# Topic distribution chart
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top 20 topics by size
top20  = topic_info[topic_info["Topic"] != -1].head(20)
labels = [f"T{row['Topic']}: {topic_labels.get(row['Topic'], '')[:20]}"
          for _, row in top20.iterrows()]
sizes  = top20["Count"].tolist()

axes[0].barh(labels, sizes, color="#3498db", alpha=0.85, edgecolor="white")
axes[0].set_title("Top 20 Topics by Size", fontweight="bold")
axes[0].set_xlabel("Post count")
axes[0].grid(axis="x", alpha=0.3)
axes[0].invert_yaxis()

# Outlier analysis
n_total   = len(topics)
n_outlier = topics.count(-1)
n_assigned= n_total - n_outlier
axes[1].pie(
    [n_assigned, n_outlier],
    labels=[f"Assigned{n_assigned:,}", f"Outlier{n_outlier:,}"],
    colors=["#3498db", "#95a5a6"],
    autopct="%1.1f%%",
    startangle=90,
)
axes[1].set_title("Topic Assignment Rate", fontweight="bold")

plt.suptitle("BERTopic — Discovered Topics", fontweight="bold")
plt.tight_layout()
out_path = os.path.join(OUTPUTS_VIZ, "08_topic_distribution.png")
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {out_path}")


Saved → /content/drive/MyDrive/brand-sentiment-monitor/outputs/visualizations/08_topic_distribution.png


## 4. Save Model

In [16]:
topic_model.save(MODEL_OUT, serialization="safetensors", save_ctfidf=True)
print(f"BERTopic saved → {MODEL_OUT}")

saved = os.listdir(MODEL_OUT)
print(f"\nmodels/topic/ — {len(saved)} files:")
for fname in sorted(saved):
    sz = os.path.getsize(os.path.join(MODEL_OUT, fname)) / 1e6
    print(f"  {fname:<40}  {sz:>6.1f} MB")


BERTopic saved → /content/drive/MyDrive/brand-sentiment-monitor/models/topic

models/topic/ — 5 files:
  config.json                                  0.0 MB
  ctfidf.safetensors                           0.9 MB
  ctfidf_config.json                           0.1 MB
  topic_embeddings.safetensors                 0.2 MB
  topics.json                                  0.8 MB


In [17]:
# ── CELL 14: Fix topic.py on Drive, then verify TopicModel.load() ─────────────
#
# WHY THIS CELL OVERWRITES topic.py FIRST:
#   The original topic.py written by notebook 00 had two bugs:
#   Bug 1 — BERTopic.load(path) missing embedding_model= argument
#           → silent load but crashes at inference time
#   Bug 2 — prob = round(float(probs[0]), 4)
#           probs from BERTopic.transform() is shape (n_docs, n_topics)
#           so probs[0] is a 1-D array of ALL topic probabilities, not
#           a single float. float() on an array raises TypeError.
#           Fix: use probs[0][topic_id] to get the assigned topic's prob.
#
# Notebook 00 skips files that already exist on Drive, so re-running it
# would NOT fix these. This cell overwrites the file directly.

import sys, os
import numpy as np

# ── Step A: overwrite src/models/topic.py on Drive ────────────────────────────
topic_py_path = os.path.join(DRIVE_ROOT, "src", "models", "topic.py")

FIXED_TOPIC_PY = '"""\nsrc/models/topic.py\nMODULE 7 — Topic Modeling Engine\n\nDiscovers what customers are talking about per brand.\nAlgorithm: BERTopic (embedding-based, no bag-of-words assumptions)\n\nTraining:  notebook 08_topic_model.ipynb\nSaved via: topic_model.save("models/topic/", serialization="safetensors")\n\nUsed by: api/predict.py, aggregation/aggregator.py\n"""\n\nfrom __future__ import annotations\nimport numpy as np\n\nMODEL_PATH = "models/topic"\n\n\nclass TopicModel:\n    """\n    Wrapper around BERTopic model.\n\n    Usage:\n        model = TopicModel.load("/path/to/models/topic")\n        result = model.predict("The sole peeled off after 3 weeks")\n        # {"topic_id": 3, "label": "sole_durability", "keywords": [...], "probability": 0.82}\n    """\n\n    def __init__(self, model):\n        self.model = model\n\n    @classmethod\n    def load(cls, path: str = MODEL_PATH) -> "TopicModel":\n        from bertopic import BERTopic\n        from sentence_transformers import SentenceTransformer\n        # REQUIRED: pass embedding_model= when loading a safetensors-serialized\n        # BERTopic model. Without this, transform() raises ValueError because\n        # BERTopic cannot encode new texts at inference time.\n        embedding_model = SentenceTransformer("all-MiniLM-L6-v2")\n        model = BERTopic.load(path, embedding_model=embedding_model)\n        return cls(model)\n\n    @staticmethod\n    def _extract_prob(probs_row, topic_id: int) -> float:\n        """\n        Safely extract the probability scalar for a single document.\n\n        BERTopic.transform() returns probs with shape (n_docs, n_topics).\n        probs[doc_idx] is therefore a 1-D array of all topic probabilities.\n        We index into it with the assigned topic_id to get one float.\n\n        Edge cases:\n          - topic_id == -1  (outlier): use max probability in the row\n          - probs is None   (some BERTopic configs skip probability calc)\n          - probs_row is already a scalar (older BERTopic versions)\n        """\n        if probs_row is None:\n            return 0.0\n        arr = np.asarray(probs_row)\n        if arr.ndim == 0:                   # already a scalar\n            return round(float(arr), 4)\n        if topic_id == -1 or topic_id >= len(arr):\n            return round(float(arr.max()), 4)\n        return round(float(arr[topic_id]), 4)\n\n    def predict(self, text: str) -> dict:\n        """\n        Assign a topic to a single text.\n\n        Returns:\n            {\n                "topic_id":    int,        # -1 = outlier / no clear topic\n                "label":       str,        # e.g. "sole_durability_quality"\n                "keywords":    list[str],  # top keywords for this topic\n                "probability": float,      # 0.0 - 1.0\n            }\n        """\n        topics, probs = self.model.transform([text])\n        topic_id = int(topics[0])\n        keywords = [w for w, _ in self.model.get_topic(topic_id)] if topic_id != -1 else []\n        label    = "_".join(keywords[:3]) if keywords else "other"\n        prob     = self._extract_prob(\n            probs[0] if probs is not None else None,\n            topic_id,\n        )\n        return {\n            "topic_id":    topic_id,\n            "label":       label,\n            "keywords":    keywords[:10],\n            "probability": prob,\n        }\n\n    def predict_batch(self, texts: list[str]) -> list[dict]:\n        topics, probs = self.model.transform(texts)\n        results = []\n        for i, (topic_id, prob_row) in enumerate(zip(topics, probs)):\n            topic_id = int(topic_id)\n            keywords = [w for w, _ in self.model.get_topic(topic_id)] if topic_id != -1 else []\n            label    = "_".join(keywords[:3]) if keywords else "other"\n            prob     = self._extract_prob(prob_row, topic_id)\n            results.append({\n                "topic_id":    topic_id,\n                "label":       label,\n                "keywords":    keywords[:10],\n                "probability": prob,\n            })\n        return results\n'

with open(topic_py_path, "w") as f:
    f.write(FIXED_TOPIC_PY)
print(f"Step A ✅  Overwrote {topic_py_path}")

# ── Step B: clear stale import cache so Python reads the new file ─────────────
for key in list(sys.modules.keys()):
    if "models.topic" in key or key == "models":
        del sys.modules[key]

from models.topic import TopicModel
print("Step B ✅  TopicModel reimported from fixed file")

# ── Step C: load the saved BERTopic model and verify predict() works ──────────
loaded_topic = TopicModel.load(MODEL_OUT)
print("Step C ✅  BERTopic loaded from Drive")
print()

verify_texts = [
    "The sole completely peeled off after three weeks of use",
    "These shoes are way too narrow and hurt my feet",
    "Really expensive for the quality you get",
    "Amazing customer service, they replaced my shoes no questions asked",
    "The color faded after the first wash",
]

print("End-to-end verification (TopicModel.load → predict):")
all_ok = True
for text in verify_texts:
    result = loaded_topic.predict(text)
    ok     = isinstance(result["topic_id"], int) and isinstance(result["label"], str)
    all_ok = all_ok and ok
    flag   = "✅" if ok else "❌"
    print(f'  {flag} Topic {result["topic_id"]:>3}  '
          f'prob={result["probability"]:.3f}  '
          f'label={result["label"][:35]}')
    print(f'       text: {text[:65]}')
    print()

if all_ok:
    print("✅ TopicModel.load() working correctly — model is ready for backend handoff")
else:
    raise RuntimeError("Verification failed — check that models/topic/ was saved in cell 13")


Step A ✅  Overwrote /content/drive/MyDrive/brand-sentiment-monitor/src/models/topic.py
Step B ✅  TopicModel reimported from fixed file


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step C ✅  BERTopic loaded from Drive

End-to-end verification (TopicModel.load → predict):


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

2026-03-04 06:51:36,354 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.


  ✅ Topic  50  prob=0.092  label=shoes_feet_boots
       text: Amazing customer service, they replaced my shoes no questions ask



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-03-04 06:51:36,385 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.


  ✅ Topic  90  prob=0.043  label=blue_pink_color
       text: The color faded after the first wash

✅ TopicModel.load() working correctly — model is ready for backend handoff


## 5. Save Evaluation Report

In [18]:
n_topics   = len(set(topics)) - (1 if -1 in topics else 0)
outlier_pct= topics.count(-1) / len(topics) * 100

topic_inventory = []
for topic_id in sorted(set(topics)):
    if topic_id == -1:
        continue
    kws = topic_model.get_topic(topic_id)
    topic_inventory.append({
        "topic_id"  : topic_id,
        "label"     : topic_labels.get(topic_id, f"topic_{topic_id}"),
        "count"     : int(topic_info[topic_info["Topic"]==topic_id]["Count"].values[0]),
        "keywords"  : [w for w, _ in kws[:10]],
        "top_scores": [round(s, 4) for _, s in kws[:10]],
    })

topic_report = {
    "model_path"         : "models/topic/",
    "algorithm"          : "BERTopic",
    "embedding_model"    : "all-MiniLM-L6-v2",
    "n_topics_discovered": n_topics,
    "n_texts_trained_on" : len(all_texts),
    "outlier_pct"        : round(outlier_pct, 2),
    "assignment_pct"     : round(100 - outlier_pct, 2),
    "topic_inventory"    : topic_inventory,
    "outputs": {
        "model_dir"     : "models/topic/",
        "distribution_png": "outputs/visualizations/08_topic_distribution.png",
        "eval_json"     : "outputs/reports/topic_eval.json",
    },
}

rpt_path = os.path.join(OUTPUTS_RPT, "topic_eval.json")
with open(rpt_path, "w") as f:
    json.dump(topic_report, f, indent=2)

print(f"Saved → {rpt_path}")
print()
print("=" * 55)
print("TOPIC MODEL — FINAL SUMMARY")
print("=" * 55)
print(f"  Algorithm    : BERTopic")
print(f"  Corpus size  : {len(all_texts):,} texts")
print(f"  Topics found : {n_topics}")
print(f"  Assigned     : {100-outlier_pct:.1f}%  of texts get a topic")
print(f"  Outlier      : {outlier_pct:.1f}%  (topic_id = -1)")
print(f"  Model saved  : models/topic/")
print("=" * 55)
print(f"\n✅ Notebook 08 complete. Next: 09_integration_test.ipynb")


Saved → /content/drive/MyDrive/brand-sentiment-monitor/outputs/reports/topic_eval.json

TOPIC MODEL — FINAL SUMMARY
  Algorithm    : BERTopic
  Corpus size  : 84,548 texts
  Topics found : 160
  Assigned     : 54.8%  of texts get a topic
  Outlier      : 45.2%  (topic_id = -1)
  Model saved  : models/topic/

✅ Notebook 08 complete. Next: 09_integration_test.ipynb
